In [1]:
import pandas as pd
import numpy as np
from keras.models import load_model
import joblib
import os

def gerar_previsoes_2016_2025():
    # 1. Carregar modelo e scaler
    # O custom_objects trata funções de perda personalizadas que podem estar no .h5
    model = load_model('best_lstm_model.h5', custom_objects={'mape_loss': lambda y_true, y_pred: 0})
    scaler = joblib.load('scaler_itapua.save')
    
    # 2. Carregar dados históricos
    path = '..'
    caminho_arquivo = os.path.join(path, 'includes', 'dados', 'Tabela_consumo_Itapua_120m.csv')
    
    if not os.path.exists(caminho_arquivo):
        print(f"Erro: Arquivo não encontrado em {caminho_arquivo}")
        return
        
    df = pd.read_csv(caminho_arquivo, sep=';')
    df['AM_REFERENCIA'] = pd.to_datetime(df['AM_REFERENCIA'], format='%Y%m')
    
    # Agrupar e ordenar
    df_aggregated = df.groupby('AM_REFERENCIA')['HCLQTCON'].sum().reset_index()
    df_aggregated = df_aggregated.sort_values(by='AM_REFERENCIA')
    
    # 3. Definir Janela e Período
    look_back = 6  # O modelo usa os últimos 6 meses para prever o próximo
    data_inicio_alvo = pd.to_datetime('2016-01-01')
    data_fim_alvo = pd.to_datetime('2025-12-01')
    
    # Criar o intervalo de datas (mensal)
    previsao_dates = pd.date_range(start=data_inicio_alvo, end=data_fim_alvo, freq='MS')
    
    # 4. Obter a "semente" (os 6 meses anteriores a janeiro de 2016)
    # Isso é necessário para gerar a primeira previsão (Jan/2016)
    dados_semente = df_aggregated[df_aggregated['AM_REFERENCIA'] < data_inicio_alvo].tail(look_back)
    
    if len(dados_semente) < look_back:
        print(f"Erro: Dados históricos insuficientes antes de 2016. Necessário {look_back} meses.")
        return

    # Normalizar a janela inicial
    current_window = scaler.transform(dados_semente[['HCLQTCON']]).flatten()
    
    # 5. Loop de Previsão Recursiva
    lista_previsoes_scaled = []
    
    print(f"Gerando previsões para o período: {data_inicio_alvo.year} a {data_fim_alvo.year}...")
    
    for i in range(len(previsao_dates)):
        # Redimensionar para o formato LSTM: [amostras, timesteps, características]
        x_input = current_window.reshape(1, look_back, 1)
        
        # Prever o próximo ponto
        pred_scaled = model.predict(x_input, verbose=0)
        
        # Guardar o resultado escalado
        lista_previsoes_scaled.append(pred_scaled[0, 0])
        
        # Atualizar a janela: remove o primeiro mês e adiciona a nova previsão no final
        current_window = np.append(current_window[1:], pred_scaled[0, 0])
    
    # 6. Desnormalizar os resultados para a escala real (Consumo)
    previsoes_finais = scaler.inverse_transform(np.array(lista_previsoes_scaled).reshape(-1, 1)).flatten()
    
    # 7. Organizar e Salvar
    df_resultado = pd.DataFrame({
        'Mes_Referencia': previsao_dates,
        'Previsao_Consumo': previsoes_finais
    })
    
    df_resultado.to_csv('previsao_consumo_2016_2025.csv', index=False, sep=';')
    print("\nArquivo 'previsao_consumo_2016_2025.csv' gerado com sucesso.")
    
    return df_resultado

# Execução
resultados = gerar_previsoes_2016_2025()
if resultados is not None:
    print("\nPrimeiras linhas da previsão:")
    print(resultados.head())
    print("\nÚltimas linhas da previsão:")
    print(resultados.tail())



Gerando previsões para o período: 2016 a 2025...

Arquivo 'previsao_consumo_2016_2025.csv' gerado com sucesso.

Primeiras linhas da previsão:
  Mes_Referencia  Previsao_Consumo
0     2016-01-01     187559.484375
1     2016-02-01     186078.250000
2     2016-03-01     181962.000000
3     2016-04-01     173836.593750
4     2016-05-01     173566.671875

Últimas linhas da previsão:
    Mes_Referencia  Previsao_Consumo
115     2025-08-01     178533.421875
116     2025-09-01     174882.453125
117     2025-10-01     173566.671875
118     2025-11-01     173566.671875
119     2025-12-01     173566.671875
